# gTCN fixed-z evaluation

This notebook loads robust stored embeddings for BritPlexi and FenderDouble. Rendering uses only the new dry guitar and the stored z; the effect encoder is bypassed. It also evaluates a custom effect using the average of the two stored z vectors.

Create the embedding files first (one per effect, e.g. `checkpoints/BritPlexi_embeddings.pt`):

```bash
python src/compute_effect_embeddings.py --checkpoint checkpoints/best.pt --output-dir checkpoints
```

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
import torchaudio
from IPython.display import Audio, display

from dataset import GuitarPairIndex
from inference import GuitarRenderer
from utils import resolve_path

TIMBRES = ['BritPlexi', 'FenderDouble']
CHECKPOINT_PATH = REPO_ROOT / 'checkpoints' / 'best.pt'
EMBEDDINGS_DIR = REPO_ROOT / 'checkpoints'
renderer = GuitarRenderer(str(CHECKPOINT_PATH))
renderer.load_effect_embeddings(str(EMBEDDINGS_DIR))
SAMPLE_RATE = renderer.sample_rate
STORED_Z = renderer.effect_embeddings
print('Renderer:', CHECKPOINT_PATH)
print('Embeddings:', EMBEDDINGS_DIR)
print({name: tuple(value.shape) for name, value in STORED_Z.items()})

## Helpers

In [ ]:
def load_audio(path, target_sr=SAMPLE_RATE):
    audio, sr = sf.read(str(path), dtype='float32', always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=-1)
    if sr != target_sr:
        audio = torchaudio.functional.resample(torch.from_numpy(audio).unsqueeze(0), sr, target_sr).squeeze(0).numpy()
    return audio

@torch.no_grad()
def render_z(dry_audio, z):
    dry = torch.from_numpy(dry_audio).float().to(renderer.device).view(1, 1, -1)
    z = z.float().to(renderer.device).view(1, -1)
    return renderer.model.render_with_z(dry, z).squeeze().cpu().numpy()

def plot_spectrograms(signals, labels):
    n_fft, hop = 2048, 512
    window = torch.hann_window(n_fft)
    fig, axes = plt.subplots(1, len(signals), figsize=(5 * len(signals), 4), sharey=True)
    axes = np.atleast_1d(axes)
    for axis, signal, label in zip(axes, signals, labels):
        spec = torch.stft(torch.from_numpy(signal).float(), n_fft=n_fft, hop_length=hop, window=window, return_complex=True)
        db = 20 * torch.log10(spec.abs().clamp(min=1e-5))
        image = axis.imshow(db.numpy(), origin='lower', aspect='auto', extent=[0, len(signal) / SAMPLE_RATE, 0, SAMPLE_RATE / 2], cmap='magma')
        axis.set_title(label)
        axis.set_xlabel('Time (s)')
    axes[0].set_ylabel('Frequency (Hz)')
    fig.colorbar(image, ax=axes, fraction=0.02, pad=0.02, label='dB')
    plt.show()

def plot_waveforms(signals, labels, max_seconds=5.0):
    n = int(SAMPLE_RATE * max_seconds)
    fig, axes = plt.subplots(len(signals), 1, figsize=(12, 2.1 * len(signals)), sharex=True)
    axes = np.atleast_1d(axes)
    for axis, signal, label in zip(axes, signals, labels):
        clipped = signal[:n]
        axis.plot(np.arange(len(clipped)) / SAMPLE_RATE, clipped, linewidth=0.6)
        axis.set_title(label)
        axis.set_ylim(-1.05, 1.05)
    axes[-1].set_xlabel('Time (s)')
    fig.tight_layout()
    plt.show()

def compare_metrics(predicted, target):
    length = min(len(predicted), len(target))
    predicted, target = predicted[:length], target[:length]
    print(f"L1: {np.mean(np.abs(predicted - target)):.4f}   RMSE: {np.sqrt(np.mean((predicted - target) ** 2)):.4f}")

def save_prediction(audio, name):
    output_dir = REPO_ROOT / 'outputs' / 'eval'
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f'{name}.wav'
    sf.write(str(output_path), audio, SAMPLE_RATE)
    print('Saved:', output_path)
    return output_path

## Compare stored-z BritPlexi and FenderDouble

In [ ]:
cfg = renderer.cfg['data']
x_root = resolve_path(cfg['x_root'])
y_root = resolve_path(cfg['y_root'])
indices = {name: GuitarPairIndex(x_root, y_root, name) for name in TIMBRES}
dry_path, brit_wet_path, _ = sorted(indices['BritPlexi'].pairs)[0]
double_by_dry = {pair[0]: pair for pair in indices['FenderDouble'].pairs}
_, double_wet_path, _ = double_by_dry[dry_path]
dry_audio = load_audio(dry_path)
wet_truth = {'BritPlexi': load_audio(brit_wet_path), 'FenderDouble': load_audio(double_wet_path)}
wet_predicted = {name: render_z(dry_audio, STORED_Z[name]) for name in TIMBRES}
print('Dry:', dry_path)
for name in TIMBRES:
    print(name, 'prediction shape:', wet_predicted[name].shape)
    compare_metrics(wet_predicted[name], wet_truth[name])

In [ ]:
for name in TIMBRES:
    plot_spectrograms([dry_audio, wet_truth[name], wet_predicted[name]], ["Dry", f"{name} truth", f"{name} fixed-z prediction"])

print('Dry:')
display(Audio(dry_audio, rate=SAMPLE_RATE))
for name in TIMBRES:
    print(name, 'truth:')
    display(Audio(wet_truth[name], rate=SAMPLE_RATE))
    print(name, 'fixed-z prediction:')
    display(Audio(wet_predicted[name], rate=SAMPLE_RATE))
    save_prediction(wet_predicted[name], f'{Path(dry_path).stem}-{name}-fixedz-predicted')

## Custom averaged effect

This keeps the same renderer and averages only the two stored effect vectors.

In [ ]:
averaged_z = (STORED_Z['BritPlexi'] + STORED_Z['FenderDouble']) / 2
averaged_prediction = render_z(dry_audio, averaged_z)
print('Averaged z shape:', averaged_z.shape)
plot_spectrograms([dry_audio, averaged_prediction], ['Dry', 'Average BritPlexi + FenderDouble'])
plot_waveforms([dry_audio, averaged_prediction], ['Dry', 'Average BritPlexi + FenderDouble'])
print('Dry:')
display(Audio(dry_audio, rate=SAMPLE_RATE))
print('Averaged-z prediction:')
display(Audio(averaged_prediction, rate=SAMPLE_RATE))
save_prediction(averaged_prediction, f'{Path(dry_path).stem}-BritPlexi_FenderDouble_avg-fixedz-predicted')

## Try a new dry guitar recording

In [ ]:
NEW_DRY_PATH = REPO_ROOT.parent / 'GTtimbre' / 'data' / 'demo' / 'demo1.wav'

new_dry_audio = load_audio(NEW_DRY_PATH)
new_wet0 = render_z(new_dry_audio, STORED_Z['BritPlexi'])
new_wet1 = render_z(new_dry_audio, STORED_Z['FenderDouble'])
new_wet2 = render_z(new_dry_audio, averaged_z)

print('Input:', NEW_DRY_PATH)

plot_spectrograms([new_dry_audio, new_wet0, new_wet1, new_wet2], ['Dry', 'BritPlexi fixed-z prediction', 'FenderDouble fixed-z prediction', 'average-z prediction'])

plot_waveforms([new_dry_audio, new_wet0, new_wet1, new_wet2], ['Dry', 'BritPlexi fixed-z prediction', 'FenderDouble fixed-z prediction', 'average-z prediction'])
display(Audio(new_dry_audio, rate=SAMPLE_RATE))
display(Audio(new_wet0, rate=SAMPLE_RATE))
display(Audio(new_wet1, rate=SAMPLE_RATE))
display(Audio(new_wet2, rate=SAMPLE_RATE))

save_prediction(new_wet0, f'{Path(NEW_DRY_PATH).stem}-BritPlexi-fixedz-predicted')
save_prediction(new_wet1, f'{Path(NEW_DRY_PATH).stem}-FenderDouble-fixedz-predicted')
save_prediction(new_wet2, f'{Path(NEW_DRY_PATH).stem}-average-z-predicted')